# Feeding data: Dataset and DataLoader

_PyTorch_

**Wrap your data in a Dataset, hand it to a DataLoader, and get batched, shuffled, augmented, parallel-loaded tensors that keep the GPU (Graphics Processing Unit) busy.**

Separate what one sample is from how samples become batches. The
       Dataset owns the first; the DataLoader owns the second. Because of that clean
       split, the same DataLoader machinery &mdash; batching, shuffling, parallel workers, pinned memory &mdash;
       works for images, text, audio, or numpy tables without change. You write a tiny Dataset; you get an
       industrial data pipeline.

---

This notebook builds that pipeline one piece at a time — a custom Dataset, the DataLoader that batches it, real image transforms, and a custom collate for ragged data. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns**. We pull a few raw samples from the MNIST dataset to see them before any transforms.

In [ ]:
import torchvision

preview = torchvision.datasets.MNIST(root="./data", train=True, download=True)
print("dataset: MNIST   samples:", len(preview))
first_image, first_label = preview[0]
print("one sample:", first_image.size, "image,  label =", first_label)
print("classes:", getattr(preview, "classes", "(digit labels 0-9)"))
samples = [preview[i] for i in range(5)]
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — A custom `Dataset` over numpy arrays

A map-style `Dataset` is defined by exactly two methods: `__len__` (how many samples there are) and `__getitem__` (return ONE `(x, y)` pair). Keep the raw numpy arrays and convert each sample to the right tensor dtypes *inside* `__getitem__` — `float32` for the features the model multiplies, and a `long` class index because `CrossEntropyLoss` wants integer targets. That tiny class is the whole contract; everything else (batching, shuffling) is the DataLoader's job.

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)          # reproducible shuffling / results
device = "cuda" if torch.cuda.is_available() else "cpu"

# A CUSTOM Dataset over numpy arrays — the most common starting point.
class ArrayDataset(Dataset):
    def __init__(self, X, y):
        # keep the raw arrays; convert per-sample in __getitem__
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)

    def __len__(self):                      # how many samples
        return len(self.X)

    def __getitem__(self, i):               # ONE (x, y) pair, correct dtypes
        x = torch.from_numpy(self.X[i])                  # float32 feature vector
        y = torch.tensor(self.y[i], dtype=torch.long)    # long class index (CrossEntropyLoss wants this)
        return x, y

# synthetic 2-class data: 500 samples, 3 features
X = np.random.randn(500, 3).astype(np.float32)
y = (X[:, 0] + X[:, 1] > 0).astype(np.int64)

ds = ArrayDataset(X, y)
print("dataset size:", len(ds), "| one sample:", ds[0][0].shape, ds[0][1].item())

### Step 2 — Hand the Dataset to a `DataLoader`

The Dataset knows what one sample is; the `DataLoader` turns samples into batches. It batches, shuffles (TRAIN only), spreads loading across worker processes, and pins memory for a faster CPU→GPU copy. Pull one batch with `next(iter(loader))` and notice the collate step stacked 64 individual `(3,)` vectors into a single `(64, 3)` tensor.

In [ ]:
# The DataLoader: batch + shuffle (TRAIN only) + parallel workers + pinned memory for GPU.
loader = DataLoader(
    ds,
    batch_size=64,
    shuffle=True,                   # shuffle the TRAIN loader only
    num_workers=2,                  # parallel worker processes prefetch batches (0 on Windows/notebooks)
    pin_memory=(device == "cuda"),  # faster CPU->GPU copy when training on GPU
    drop_last=False,                # keep the final short batch
)

xb, yb = next(iter(loader))         # grab ONE batch
print("batch x:", xb.shape, xb.dtype, "| batch y:", yb.shape, yb.dtype)
# batch x: torch.Size([64, 3]) torch.float32 | batch y: torch.Size([64]) torch.int64

### Step 3 — A real image dataset with `transforms` (train vs eval hygiene)

`torchvision` datasets plug into the same DataLoader machinery. The key discipline is two separate transform pipelines: the TRAIN transform applies random augmentation (here a random crop) *before* `ToTensor` and `Normalize`, while the EVAL transform is deterministic — same normalization, no randomness. Augmenting the eval set would make the metric noisy; that is why we keep them apart and only shuffle the train loader.

In [ ]:
# TORCHVISION MNIST with transforms.Compose + a DataLoader.
from torchvision import datasets, transforms

# TRAIN transform: augment, then ToTensor (-> [0,1]), then Normalize (MNIST mean/std).
train_tf = transforms.Compose([
    transforms.RandomCrop(28, padding=2),       # augmentation: TRAIN ONLY
    transforms.ToTensor(),                      # PIL/np uint8 [0,255] -> float tensor [0,1], shape (1,28,28)
    transforms.Normalize((0.1307,), (0.3081,)), # zero-center using THIS dataset's stats
])

# EVAL transform: deterministic only -- no random augmentation, same Normalize.
eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])

train_ds = datasets.MNIST(root="./data", train=True,  download=True, transform=train_tf)
test_ds  = datasets.MNIST(root="./data", train=False, download=True, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,
                          num_workers=2, pin_memory=(device == "cuda"))
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False,   # eval: no shuffle
                          num_workers=2, pin_memory=(device == "cuda"))

imgs, labels = next(iter(train_loader))
print("MNIST batch:", imgs.shape, imgs.dtype, "| labels:", labels.shape)
# MNIST batch: torch.Size([128, 1, 28, 28]) torch.float32 | labels: torch.Size([128])

### Step 4 — A custom `collate_fn` for variable-length data

The default collate calls `torch.stack`, which fails when samples have different shapes — like sentences of different lengths. A `collate_fn` takes the list of samples for one batch and assembles them however you like. Here `pad_sequence` pads every sequence up to the batch's longest length so they stack into one rectangle, and we also return the true lengths so the model can ignore the padding.

In [ ]:
# A CUSTOM collate_fn for VARIABLE-LENGTH data (e.g. sentences).
from torch.nn.utils.rnn import pad_sequence

class SeqDataset(Dataset):
    def __init__(self, seqs, labels):
        self.seqs, self.labels = seqs, labels
    def __len__(self):  return len(self.seqs)
    def __getitem__(self, i):
        return torch.tensor(self.seqs[i], dtype=torch.long), self.labels[i]

def pad_collate(batch):
    seqs, labels = zip(*batch)                 # lists of (seq_tensor, label)
    lengths = torch.tensor([len(s) for s in seqs])
    padded  = pad_sequence(seqs, batch_first=True, padding_value=0)  # (B, max_len)
    return padded, lengths, torch.tensor(labels)

seq_ds = SeqDataset([[1, 2, 3], [4, 5], [6, 7, 8, 9]], [0, 1, 0])
seq_loader = DataLoader(seq_ds, batch_size=3, collate_fn=pad_collate)

padded, lengths, lbls = next(iter(seq_loader))
print("padded sequences:", padded.shape, "| true lengths:", lengths.tolist())
# padded sequences: torch.Size([3, 4]) | true lengths: [3, 2, 4]

## Visualize the data & results

_Why does DataLoader parallelism matter? As we add worker processes, how does training throughput (samples/sec) change? Modelled on a small 2-conv-layer CNN (Convolutional Neural Network) whose GPU step takes ~20 ms per batch of 64, while one CPU loader needs ~115 ms of decode+augment per batch._

### Step 1 — Model the data-loading cost

To see *why* worker processes matter, we model a small CNN whose GPU step is fixed at ~20 ms per batch — that is the throughput ceiling. One CPU loader needs ~115 ms to decode and augment that same batch, so a single loader leaves the GPU starved. Each extra worker splits the CPU work, minus a small per-worker IPC overhead.

In [ ]:
import numpy as np

# Small illustrative model: a 2-conv-layer CNN. Per BATCH of 64 samples,
# the GPU forward+backward step takes ~20 ms (GPU-bound floor), while ONE
# CPU loader needs ~115 ms to decode+augment that batch. Extra workers split
# the CPU work; a small per-worker IPC overhead is added.
batch_size = 64
t_gpu      = 0.020     # 20 ms GPU step per batch  -> ceiling = 64/0.020 = 3200 samples/s
t_cpu_one  = 0.115     # 115 ms CPU decode+augment per batch with a single loader
overhead   = 0.0009    # ~0.9 ms extra per additional worker (process / IPC cost)

### Step 2 — Sweep the worker count and read the throughput

Now vary the number of workers. For each count we compute the per-batch CPU time, then take the slower of GPU vs CPU (they overlap, so the bottleneck wins) and convert to samples/sec. Throughput climbs as workers are added and then plateaus once the CPU is fast enough to keep the GPU saturated at its 3200 samples/s ceiling.

In [ ]:
workers = [0, 1, 2, 3, 4, 6, 8, 12]
print(" w  throughput(samples/s)  per-batch(ms)")
for w in workers:
    eff = max(w, 1)                               # 0 workers == main process == 1 loader
    t_cpu  = t_cpu_one / eff + overhead * max(w - 1, 0)
    t_batch = max(t_gpu, t_cpu)                   # GPU and prefetch overlap -> the slower one wins
    thpt = batch_size / t_batch
    print(f"{w:2d}  {thpt:18.0f}  {t_batch*1000:12.1f}")
# w  throughput(samples/s)  per-batch(ms)
#  0                 557          115.0
#  1                 557          115.0
#  2                1096           58.4
#  3                1595           40.1
#  4                2035           31.4
#  6                2704           23.7
#  8                3096           20.7
# 12                3200           20.0   <- plateau at the 3200/s GPU ceiling

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Type this in Colab. Write a minimal custom Dataset over two numpy arrays. With
            X = np.arange(20).reshape(10, 2).astype("float32") and y = np.arange(10),
            implement __len__ and __getitem__ (return x as
            float32, y as a long scalar). Print len(ds) and
            ds[3].

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Subclass Dataset and implement __len__ + __getitem__. — _Those two methods are the entire map-style Dataset contract._
- Cast inside __getitem__: input torch.float32, label torch.long. — _The model wants float32 features; CrossEntropyLoss wants a long class index._

**Answer:** import numpy as np, torch
from torch.utils.data import Dataset, DataLoader

class ArrayDS(Dataset):
    def __init__(self, X, y):
        self.X, self.y = X, y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, i):
        x = torch.tensor(self.X[i], dtype=torch.float32)
        t = torch.tensor(self.y[i], dtype=torch.long)
        return x, t

X = np.arange(20).reshape(10, 2).astype("float32")
y = np.arange(10)
ds = ArrayDS(X, y)
print(len(ds))     # 10
print(ds[3])       # (tensor([6., 7.]), tensor(3))

</details>

**Problem 2.** Type this in Colab. Wrap the ds from above in a
            DataLoader(ds, batch_size=4, shuffle=False). Grab one batch with
            next(iter(loader)). Predict the shapes and dtypes of xb and yb
            before running, then verify.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Build the DataLoader and pull one batch with next(iter(loader)). — _The loader collates 4 samples into one batched tensor._
- Print xb.shape, xb.dtype and yb.shape, yb.dtype. — _The default collate stacks the 4 (2,) inputs into (4, 2) and the 4 labels into (4,)._

**Answer:** loader = DataLoader(ds, batch_size=4, shuffle=False)
xb, yb = next(iter(loader))
print(xb.shape, xb.dtype)   # torch.Size([4, 2]) torch.float32
print(yb.shape, yb.dtype)   # torch.Size([4]) torch.int64
print(yb)                   # tensor([0, 1, 2, 3])

</details>

**Problem 3.** Type this in Colab. Show what shuffle does. Build a DataLoader over the
            same ds with batch_size=10, shuffle=False and print the label batch; then with
            shuffle=True after torch.manual_seed(0) and print it again.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set shuffle=False for the natural order, shuffle=True for a random permutation. — _shuffle=True uses a fresh random index order each epoch — for the TRAIN loader only._
- Seed with torch.manual_seed(0) before the shuffled loader. — _So the random order is reproducible for grading._

**Answer:** print(next(iter(DataLoader(ds, batch_size=10, shuffle=False)))[1])
# tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
torch.manual_seed(0)
print(next(iter(DataLoader(ds, batch_size=10, shuffle=True)))[1])
# tensor([4, 1, 7, 5, 3, 9, 0, 8, 6, 2])  -- a random permutation

</details>

**Problem 4.** Type this in Colab. Show drop_last. With 10 samples and batch_size=4, iterate
            the loader once with drop_last=False and print each batch size; then with
            drop_last=True. Predict the batch counts before running.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Iterate the loader and print len(xb) per batch. — _10 / 4 leaves a final short batch of 2._
- Compare drop_last=False (keeps it: 4,4,2) vs drop_last=True (drops it: 4,4). — _drop_last=True guarantees every batch is exactly batch_size._

**Answer:** for drop in (False, True):
    dl = DataLoader(ds, batch_size=4, drop_last=drop)
    print(drop, [len(xb) for xb, _ in dl])
# False [4, 4, 2]

</details>

**Problem 5.** Type this in Colab. The dtype pitfall. Make a Dataset whose __getitem__ wrongly returns
            (self.X[i], int(self.y[i])) with X a numpy float64 array. Feed a
            batch to nn.Linear(2, 3) + nn.CrossEntropyLoss and observe the dtype error. Then
            fix __getitem__ to return float32 input and a long target.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Return real tensors: input torch.float32, label torch.long (not a Python int or float64). — _Linear weights are float32 and CrossEntropyLoss needs a long index tensor._
- Run a forward+loss on one batch after the fix. — _It confirms the corrected dtypes flow cleanly through model and loss._

**Answer:** import torch.nn as nn
Xf = np.arange(20).reshape(10, 2).astype("float64")   # float64 -> mismatch
class GoodDS(Dataset):
    def __init__(s, X, y): s.X, s.y = X, y
    def __len__(s): return len(s.X)
    def __getitem__(s, i):
        return (torch.tensor(s.X[i], dtype=torch.float32),   # fix: float32
                torch.tensor(s.y[i], dtype=torch.long))      # fix: long index
dl = DataLoader(GoodDS(Xf, np.arange(10) % 3), batch_size=4)
xb, yb = next(iter(dl))
loss = nn.CrossEntropyLoss()(nn.Linear(2, 3)(xb), yb)
print(loss.shape, xb.dtype, yb.dtype)   # torch.Size([]) torch.float32 torch.int64

</details>

**Problem 6.** Type this in Colab. Use a built-in dataset split into train and eval loaders with the RIGHT hygiene.
            Build a train transform Compose([RandomHorizontalFlip(), ToTensor(), Normalize((0.5,),(0.5,))])
            and a deterministic eval transform Compose([ToTensor(), Normalize((0.5,),(0.5,))]). Make a
            train DataLoader(shuffle=True) and a test DataLoader(shuffle=False) over MNIST.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Give train and eval separate transforms — augment train only, deterministic eval. — _Random crops/flips on eval make the metric noisy; eval must be deterministic._
- Shuffle the train loader, not the eval loader. — _Shuffling decorrelates train batches; eval should be a fixed, reproducible pass._

**Answer:** from torchvision import datasets, transforms
train_tf = transforms.Compose([
    transforms.RandomHorizontalFlip(),          # augmentation: TRAIN only
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
eval_tf = transforms.Compose([                  # deterministic: no randomness
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)),
])
train_ds = datasets.MNIST("./data", train=True,  download=True, transform=train_tf)
test_ds  = datasets.MNIST("./data", train=False, download=True, transform=eval_tf)
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True)    # shuffle TRAIN
test_dl  = DataLoader(test_ds,  batch_size=256, shuffle=False)   # NOT eval
imgs, labels = next(iter(train_dl))
print(imgs.shape, labels.shape)   # torch.Size([128, 1, 28, 28]) torch.Size([128])

</details>

**Problem 7.** Type this in Colab. Variable-length data needs a custom collate_fn. Build a Dataset of
            sequences [[1,2,3], [4,5], [6,7,8,9]] with labels [0,1,0] returning each as a
            long tensor. Write pad_collate using
            nn.utils.rnn.pad_sequence(..., batch_first=True) and print the padded batch shape.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Pass collate_fn=pad_collate to the DataLoader. — _The default collate calls torch.stack, which fails on different-length sequences._
- Pad with pad_sequence(seqs, batch_first=True, padding_value=0). — _It pads every sequence to the batch's longest length so they stack into one rectangle._

**Answer:** from torch.nn.utils.rnn import pad_sequence
class SeqDS(Dataset):
    def __init__(s, seqs, labels): s.seqs, s.labels = seqs, labels
    def __len__(s): return len(s.seqs)
    def __getitem__(s, i):
        return torch.tensor(s.seqs[i], dtype=torch.long), s.labels[i]

def pad_collate(batch):
    seqs, labels = zip(*batch)
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, torch.tensor(labels)

dl = DataLoader(SeqDS([[1,2,3],[4,5],[6,7,8,9]], [0,1,0]),
                batch_size=3, collate_fn=pad_collate)
padded, lbls = next(iter(dl))
print(padded.shape)   # torch.Size([3, 4])  -- padded to the longest (4)
print(padded)
# tensor([[1, 2, 3, 0],
#         [4, 5, 0, 0],
#         [6, 7, 8, 9]])

</details>

**Problem 8.** Type this in Colab. Tie it together: iterate a full epoch. Build a TensorDataset from
            X = torch.randn(50, 4), y = torch.randint(0, 2, (50,)), wrap it in a
            DataLoader(batch_size=16, shuffle=True), loop over it once, and count how many batches you get
            and the total number of samples seen.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Use TensorDataset as a quick Dataset over in-memory tensors. — _No custom class needed when the data already fits in tensors._
- Loop for xb, yb in loader: and tally len(xb). — _50 / 16 gives 4 batches (16,16,16,2) totalling 50 samples._

**Answer:** from torch.utils.data import TensorDataset
torch.manual_seed(0)
X = torch.randn(50, 4); y = torch.randint(0, 2, (50,))
loader = DataLoader(TensorDataset(X, y), batch_size=16, shuffle=True)
n_batches = total = 0
for xb, yb in loader:
    n_batches += 1
    total += len(xb)
print(n_batches, total)   # 4 50

</details>